In [2]:

import warnings
warnings.filterwarnings("ignore")
import sys
import os
from arch import arch_model
import numpy as np 
import pandas as pd
import torch
import math
from statsmodels.tsa.stattools import adfuller
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, r2_score
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import norm
from scipy import stats
import scipy.stats as scipy_stats
import plotly.graph_objects as go
from pathlib import Path

In [3]:
import importlib

workspace_root = Path.cwd()
if not (workspace_root / "ultility").exists():
    workspace_root = workspace_root.parent
for p in [workspace_root, workspace_root / "model"]:
    if p.exists():
        p_str = str(p)
        if p_str not in sys.path:
            sys.path.insert(0, p_str)

import ultility.data_loader
import ultility.metrics
import ultility.models_garch
import ultility.models_transformer
import train_pipeline
import ultility.var_calculator as var_calculator
import ultility.transformer_garch
importlib.reload(ultility.data_loader)
importlib.reload(ultility.metrics)
importlib.reload(ultility.models_garch)
importlib.reload(ultility.models_transformer)
importlib.reload(train_pipeline)
importlib.reload(ultility.var_calculator)

<module 'ultility.var_calculator' from 'd:\\UIT_LEARNING_MATERIAL\\07.Research\\code\\ultility\\var_calculator.py'>

## Market Return

The market return is calculated as the **market-cap weighted average** of individual stock returns:

$$R_t^{\text{market}} = \sum_{i=1}^{N} w_{i,t} \cdot r_{i,t}$$

where:
- $R_t^{\text{market}}$ = Market return at time $t$
- $N$ = Total number of stocks in the index
- $w_{i,t}$ = Weight of stock $i$ at time $t$
- $r_{i,t}$ = Return of stock $i$ at time $t$

The weight of each stock is based on its **market capitalization**:

$$w_{i,t} = \frac{\text{MC}_{i,t}}{\sum_{j=1}^{N} \text{MC}_{j,t}}$$

where:
- $\text{MC}_{i,t}$ = Market capitalization of stock $i$ at time $t$
- $\sum_{j=1}^{N} \text{MC}_{j,t}$ = Total market capitalization of all stocks

**Key Properties:**
- $\sum_{i=1}^{N} w_{i,t} = 1$ (weights sum to 100%)
- $w_{i,t} \geq 0$ for all $i, t$ (non-negative weights)
- Larger companies have higher influence on market return

In [4]:
path = "../dataset/VN30_dataset_from_2019.csv"
df = pd.read_csv(path)

df["time"] = pd.to_datetime(df["time"], format="mixed", dayfirst=True, errors="coerce")


df_2020 = df[df["time"].dt.year >= 2020]


stocks_returns = (
    df_2020
    .sort_values("time")
    .groupby("time")
    .apply(lambda x: np.average(
        x["return_1d"],
        weights=x["Market Capital (Bn VND)"] / x["Market Capital (Bn VND)"].sum()
    ))
    .dropna()
)

stocks_returns = stocks_returns.to_frame(name="stocks_return") 
stocks_returns["stocks_return"] = stocks_returns["stocks_return"] * 100





### Distribution of Return Series

The distribution of the return series was examined by fitting both a Student’s t-distribution and a normal distribution. The Kolmogorov–Smirnov test results indicate that the Student’s t-distribution provides a good fit to the data (p-value = 0.674), while the normal distribution is strongly rejected (p-value ≈ 0). 

These findings suggest that the return series exhibits heavy-tailed behavior, a common stylized fact in financial markets. Therefore, the Student’s t-distribution is more appropriate for modeling the return distribution in this dataset.

In [5]:
def distribution_analysis(data, name):
    returns = data.dropna().values
    
    nu, loc, scale = stats.t.fit(returns)
    ks_stat, ks_p = stats.kstest(returns, "t", args=(nu, loc, scale))
    
    norm_loc, norm_scale = stats.norm.fit(returns)
    ks_stat_norm, ks_p_norm = stats.kstest(returns, "norm", args=(norm_loc, norm_scale))
    return nu, loc, scale, ks_stat, ks_p, ks_stat_norm, ks_p_norm

datasets = {}

datasets['Stock returns'] = stocks_returns["stocks_return"]

df_vn30 = pd.read_csv("../dataset/vn30_index.csv")
df_vn30["time"] = pd.to_datetime(df_vn30["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn30_2020 = df_vn30[df_vn30["time"].dt.year >= 2020].copy()
vn30_returns = df_vn30_2020['return_1_day']
datasets['VN30 Index'] = vn30_returns

df_vn = pd.read_csv("../dataset/vn_index.csv")  
df_vn["time"] = pd.to_datetime(df_vn["time"], format="mixed", dayfirst=True, errors="coerce")
df_vn_2020 = df_vn[df_vn["time"].dt.year >= 2020].copy()
vn_returns = df_vn_2020['return_1_day'] * 100
datasets['VN Index'] = vn_returns

for file in ['DAX_40.csv', 'EuroNext_100.csv', 'IBEX_35.csv', 'KOSPI_index.csv', 'SMI.csv', 'snp500.csv', 'Nikkei_225.csv']:
    df_temp = pd.read_csv(f"../dataset/{file}")
    df_temp["time"] = pd.to_datetime(df_temp["time"], format="mixed", dayfirst=True, errors="coerce")
    df_temp_2020 = df_temp[df_temp["time"].dt.year >= 2020].copy()
    returns = np.log(df_temp_2020['close'] / df_temp_2020['close'].shift(1)).fillna(0) * 100
    datasets[file.replace('.csv', '')] = returns

summary_results = []
for name, data in datasets.items():
    nu, loc, scale, ks_t, p_t, ks_n, p_n = distribution_analysis(data, name)
    summary_results.append({
        'Dataset': name,
        'Nu': f"{nu:.4f}",
        'Location': f"{loc:.4f}",
        'Scale': f"{scale:.4f}",
        'KS_StudentT': f"{ks_t:.4f}",
        'P_StudentT': f"{p_t:.4f}",
        'KS_Normal': f"{ks_n:.4f}",
        'P_Normal': f"{p_n:.4f}",
        'Best_Fit': 'Student-t' if p_t > p_n else 'Normal'
    })

summary_df = pd.DataFrame(summary_results)
print(f"\n{'='*80}")
print("DISTRIBUTION ANALYSIS SUMMARY")
print(f"{'='*80}")
print(summary_df.to_string(index=False))
print(f"{'='*80}")


DISTRIBUTION ANALYSIS SUMMARY
      Dataset     Nu Location  Scale KS_StudentT P_StudentT KS_Normal P_Normal  Best_Fit
Stock returns 2.5164   0.1604 0.7706      0.0186     0.6738    0.1050   0.0000 Student-t
   VN30 Index 2.4411   0.0015 0.0078      0.0224     0.4317    0.1101   0.0000 Student-t
     VN Index 2.3933   0.1678 0.7175      0.0255     0.2788    0.1137   0.0000 Student-t
       DAX_40 2.9035   0.0895 0.7641      0.0156     0.8479    0.0951   0.0000 Student-t
 EuroNext_100 2.9521   0.0859 0.7003      0.0172     0.7479    0.0916   0.0000 Student-t
      IBEX_35 3.5449   0.0954 0.8219      0.0228     0.3926    0.0832   0.0000 Student-t
  KOSPI_index 4.4005   0.0844 0.9448      0.0178     0.7329    0.0537   0.0004 Student-t
          SMI 3.3731   0.0524 0.6264      0.0151     0.8780    0.0774   0.0000 Student-t
       snp500 2.8413   0.1009 0.7615      0.0156     0.8530    0.0946   0.0000 Student-t
   Nikkei_225 4.1398   0.0767 0.9841      0.0116     0.9875    0.0594   0.0001 

### Kolmogorov–Smirnov Test

The Kolmogorov–Smirnov (KS) test is a non-parametric statistical test used to determine whether two samples follow the same probability distribution, or whether a sample follows a specific theoretical distribution.

The test compares the **empirical cumulative distribution functions (ECDFs)** of two datasets and calculates the maximum absolute difference between them:

D = sup |F₁(x) − F₂(x)|

where:

- F₁(x) and F₂(x) are the cumulative distribution functions of the two samples  
- D is the KS statistic, representing the largest distance between the two distributions  

A **p-value** is then computed to evaluate the statistical significance of this difference.

- A **high p-value** indicates that the null hypothesis cannot be rejected, meaning the two samples likely come from the same distribution.  
- A **low p-value** suggests that the distributions are significantly different.

In this study, the KS test is used to ensure that the **training, validation, and testing datasets have similar return distributions**, which helps maintain consistency and reliability in model evaluation.

### Data Splitting Method

The dataset is divided into three subsets: training, validation, and testing. Instead of using a fixed ratio, the split points are determined by searching for the partition that maximizes the similarity of return distributions across the three subsets.

To achieve this, the Kolmogorov–Smirnov (KS) test is applied to measure the distributional similarity between the training–validation, training–test, and validation–test samples. For each possible split within predefined ranges, the average KS p-value across these three comparisons is computed. The split that yields the highest average p-value is selected as the optimal partition.

This approach ensures that the training, validation, and test sets share similar statistical distributions, thereby reducing the risk of distributional bias and improving the reliability of model evaluation.

In [5]:
def ks_optimal_split(series):

    returns = series.values
    n = len(returns)

    best_score = -1
    best_split = None

    for i in range(int(n*0.5), int(n*0.7)):
        for j in range(i + int(n*0.1), int(n*0.9)):

            train = returns[:i]
            val   = returns[i:j]
            test  = returns[j:]

            ks_tv = stats.ks_2samp(train, val).pvalue
            ks_tt = stats.ks_2samp(train, test).pvalue
            ks_vt = stats.ks_2samp(val, test).pvalue

            score = (ks_tv + ks_tt + ks_vt) / 3

            if score > best_score:
                best_score = score
                best_split = (i, j)

    i, j = best_split

    train = returns[:i]
    val   = returns[i:j]
    test  = returns[j:]

    return train, val, test, i, j, best_score



#train, val, test, i, j, score = ks_optimal_split(stocks_returns["stocks_return"])

In [ ]:
results = []
for name, data in datasets.items():
    train, val, test, i, j, score = ks_optimal_split(data.dropna())
    results.append({
        'Dataset': name,
        'Train': len(train),
        'Val': len(val),
        'Test': len(test),
        'KSScore': score
,    })
summary_split = pd.DataFrame(results)
print("\n" + "="*80)
print("KS-OPTIMAL DATA SPLIT SUMMARY")
print("="*80)
print(summary_split.to_string(index=False))
print("="*80)

In [10]:
summary_split

,Dataset,Train,Val,Test,KSScore
0,Stock returns,946,184,368,0.573188
1,VN30 Index,886,279,334,0.633911
2,VN Index,949,209,341,0.680706
3,DAX_40,1066,276,186,0.667419
4,EuroNext_100,769,154,615,0.477094
5,IBEX_35,962,159,421,0.375380
6,KOSPI_index,979,161,331,0.508218
7,SMI,1055,168,287,0.598059
8,snp500,1054,229,225,0.362910
9,Nikkei_225,1021,189,255,0.930103


In [6]:
data = {
    "Dataset": [
        "Stock returns",
        "VN30 Index",
        "VN Index",
        "DAX_40",
        "EuroNext_100",
        "IBEX_35",
        "KOSPI_index",
        "SMI",
        "snp500",
        "Nikkei_225",
    ],
    "Train": [946, 886, 949, 1066, 769, 962, 979, 1055, 1054, 1021],
    "Val": [184, 279, 209, 276, 154, 159, 161, 168, 229, 189],
    "Test": [368, 334, 341, 186, 615, 421, 331, 287, 225, 255],
    "KSScore": [0.573188, 0.633911, 0.680706, 0.667419, 0.477094, 0.375380, 0.508218, 0.598059, 0.362910, 0.930103],
}

df = pd.DataFrame(data)
print(df)

# DataFrame used by training code
split_df = df.set_index("Dataset")[["Train", "Val", "Test"]]
print("\n" + "=" * 80)
print("KS SPLIT LENGTHS (FOR TRAINING)")
print("=" * 80)
print(split_df)

         Dataset  Train  Val  Test   KSScore
0  Stock returns    946  184   368  0.573188
1     VN30 Index    886  279   334  0.633911
2       VN Index    949  209   341  0.680706
3         DAX_40   1066  276   186  0.667419
4   EuroNext_100    769  154   615  0.477094
5        IBEX_35    962  159   421  0.375380
6    KOSPI_index    979  161   331  0.508218
7            SMI   1055  168   287  0.598059
8         snp500   1054  229   225  0.362910
9     Nikkei_225   1021  189   255  0.930103

KS SPLIT LENGTHS (FOR TRAINING)
               Train  Val  Test
Dataset                        
Stock returns    946  184   368
VN30 Index       886  279   334
VN Index         949  209   341
DAX_40          1066  276   186
EuroNext_100     769  154   615
IBEX_35          962  159   421
KOSPI_index      979  161   331
SMI             1055  168   287
snp500          1054  229   225
Nikkei_225      1021  189   255


## LSTM-GARCH Model Architecture

The LSTM-GARCH hybrid model integrates traditional econometric GARCH volatility modeling with Long Short-Term Memory (LSTM) networks to capture both classical volatility clustering patterns and complex nonlinear dependencies in financial time series.

### Model Framework

The architecture combines two main components: an enhanced GARCH foundation that models volatility clustering and leveraged effects, and an LSTM memory mechanism that captures nonlinear temporal patterns. The integration allows the model to adaptively adjust volatility forecasts based on learned market dynamics.

### Enhanced GARCH Component with HAR Features

The base volatility equation extends the classical GARCH(1,1) formulation with Heterogeneous Autoregressive (HAR) components:

$$\text{base\_var}_t = \omega + \alpha \varepsilon_{t-1}^2 + \beta \sigma_{t-1}^2 + \lambda \varepsilon_{t-1}^2 \mathbf{I}(\varepsilon_{t-1} < 0) + \phi_1 RV_{1,t-1} + \phi_5 RV_{5,t-1} + \phi_{20} RV_{20,t-1}$$

where:
- $\omega$ represents the unconditional variance
- $\alpha$ and $\beta$ are standard ARCH and GARCH coefficients  
- $\lambda$ captures leverage effects
- $RV_{1,t} = \varepsilon_{t-1}^2$, $RV_{5,t}$ = 5-day average of squared returns, $RV_{20,t}$ = 20-day average of squared returns
- $\phi_1, \phi_5, \phi_{20}$ are HAR coefficients with normalized constraints

### LSTM Memory Mechanism

The LSTM component processes standardized inputs and applies element-wise corrections:

**Input Features:**
- Standardized shock: $s_t = \frac{\varepsilon_{t-1}}{\sqrt{\sigma_{t-1}^2 + \epsilon}}$
- Log variance: $v_t = \log(\text{base\_var}_t + \epsilon)$

**LSTM Gates (Element-wise Operations):**
$$f_t = \sigma(W_f \odot s_t + U_f \odot v_t + b_f)$$
$$i_t = \sigma(W_i \odot s_t + U_i \odot v_t + b_i)$$
$$\tilde{C}_t = \tanh(W_c \odot s_t + U_c \odot v_t + b_c)$$

where $\odot$ denotes element-wise multiplication.

**Cell State Evolution:**
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**Output Correction:**
$$\text{correction}_t = \tanh(C_t^T v)$$

### Hybrid Integration

The final volatility specification combines GARCH dynamics with multiplicative LSTM adjustments:

$$\sigma_t^2 = \text{base\_var}_t \times (1 + 0.05 \times \tanh(\text{correction}_t))$$

### Statistical Distribution

The model assumes returns follow a Student-t distribution with learnable degrees of freedom:

$$r_t | \mathcal{F}_{t-1} \sim \text{Student-t}(\nu, 0, \sigma_t^2)$$

where $\nu \geq 4$ is constrained to ensure finite fourth moments.

### Optimization Objective

The model is optimized using a composite loss function:

$$\mathcal{L} = \underbrace{\mathbb{E}[\log(\sigma_t^2) + \frac{\varepsilon_t^2}{\sigma_t^2}]}_{\text{QLIKE}} + \underbrace{0.05 \times \mathbb{E}[\text{ReLU}(\varepsilon_t^2 - \sigma_t^2)^2]}_{\text{Spike Penalty}}$$

This loss function combines quasi-maximum likelihood estimation with regularization to prevent extreme volatility spikes.

In [7]:
    class LSTMGARCH(nn.Module):
        def __init__(self, hidden_dim=16):
            super().__init__()
            self.hidden_dim = hidden_dim
            self.raw_omega = nn.Parameter(torch.tensor(-5.0))
            self.raw_alpha = nn.Parameter(torch.tensor(-2.0))
            self.raw_beta = nn.Parameter(torch.tensor(0.0))
            self.raw_lambda = nn.Parameter(torch.tensor(-2.0))
            self.raw_phi1 = nn.Parameter(torch.tensor(-1.0))
            self.raw_phi5 = nn.Parameter(torch.tensor(-1.0))
            self.raw_phi20 = nn.Parameter(torch.tensor(-1.0))
            self.Wf = nn.Parameter(torch.randn(self.hidden_dim))
            self.Uf = nn.Parameter(torch.randn(self.hidden_dim))
            self.bf = nn.Parameter(torch.zeros(self.hidden_dim))
            self.Wi = nn.Parameter(torch.randn(self.hidden_dim))
            self.Ui = nn.Parameter(torch.randn(self.hidden_dim))
            self.bi = nn.Parameter(torch.zeros(self.hidden_dim))
            self.Wc = nn.Parameter(torch.randn(self.hidden_dim))
            self.Uc = nn.Parameter(torch.randn(self.hidden_dim))
            self.bc = nn.Parameter(torch.zeros(self.hidden_dim))
            self.v = nn.Parameter(torch.randn(self.hidden_dim))
            self.w = nn.Parameter(torch.tensor(0.0))
            self.raw_nu = nn.Parameter(torch.tensor(4.0))

        def garch_params(self):
            omega = F.softplus(self.raw_omega)
            alpha = torch.sigmoid(self.raw_alpha)
            beta = torch.sigmoid(self.raw_beta)
            scale = alpha + beta + 1e-6
            alpha = alpha / scale * 0.95
            beta = beta / scale * 0.95
            lambda_ = torch.sigmoid(self.raw_lambda)
            phi1 = torch.sigmoid(self.raw_phi1)
            phi5 = torch.sigmoid(self.raw_phi5)
            phi20 = torch.sigmoid(self.raw_phi20)
            phi_sum = phi1 + phi5 + phi20 + 1e-6
            phi1 = phi1 / phi_sum * 0.6
            phi5 = phi5 / phi_sum * 0.3
            phi20 = phi20 / phi_sum * 0.1
            return omega, alpha, beta, lambda_, phi1, phi5, phi20

        def student_nu(self):
            return torch.clamp(F.softplus(self.raw_nu) + 2, min=4)

        def forward(self, returns):
            omega, alpha, beta, lambda_, phi1, phi5, phi20 = self.garch_params()
            nu = self.student_nu()
            batch_size, T = returns.shape
            sigma2_list = []
            c_t = torch.zeros(batch_size, self.hidden_dim, device=returns.device)
            sigma2_t = returns[:, 0] ** 2 + 1e-6
            sigma2_list.append(sigma2_t)
            squared_returns_history = [returns[:, 0] ** 2]
            for t in range(1, T):
                eps_prev = returns[:, t - 1]
                shock = eps_prev / torch.sqrt(sigma2_t + 1e-8)
                neg = (eps_prev < 0).float()
                leverage = lambda_ * eps_prev ** 2 * neg
                RV1 = eps_prev ** 2
                if len(squared_returns_history) >= 5:
                    RV5 = torch.stack(squared_returns_history[-5:], dim=1).mean(dim=1)
                else:
                    RV5 = torch.stack(squared_returns_history, dim=1).mean(dim=1)
                if len(squared_returns_history) >= 20:
                    RV20 = torch.stack(squared_returns_history[-20:], dim=1).mean(dim=1)
                else:
                    RV20 = torch.stack(squared_returns_history, dim=1).mean(dim=1)
                base_var = omega + alpha * eps_prev ** 2 + beta * sigma2_t + leverage + phi1 * RV1 + phi5 * RV5 + phi20 * RV20
                log_sigma = torch.log(base_var + 1e-8)
                shock = shock.unsqueeze(1)
                log_sigma = log_sigma.unsqueeze(1)
                f_t = torch.sigmoid(self.Wf * shock + self.Uf * log_sigma + self.bf)
                i_t = torch.sigmoid(self.Wi * shock + self.Ui * log_sigma + self.bi)
                c_hat = torch.tanh(self.Wc * shock + self.Uc * log_sigma + self.bc)
                c_t = f_t * c_t + i_t * c_hat
                correction = torch.tanh((c_t * self.v).sum(dim=1))
                sigma2_t = base_var * (1 + 0.05 * torch.tanh(correction))
                sigma2_t = torch.clamp(sigma2_t, min=1e-8)
                sigma2_list.append(sigma2_t)
                squared_returns_history.append(eps_prev ** 2)
            sigma2 = torch.stack(sigma2_list, dim=1)
            eps = returns / torch.sqrt(sigma2)
            term1 = 0.5 * torch.log(sigma2)
            term2 = (nu + 1) / 2 * torch.log(1 + eps ** 2 / ((nu - 2) * sigma2))
            const = torch.lgamma((nu + 1) / 2) - torch.lgamma(nu / 2) - 0.5 * torch.log((nu - 2) * math.pi)
            nll = term1 + term2 - const
            return nll.mean(), sigma2


Training the LSTM - GARCH models

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
seq_len = 60

def create_sequences(data, seq_len):
    xs = []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
    return np.array(xs, dtype=np.float32)

def compute_metrics(realized_vol, predicted_vol, realized_var, predicted_var):
    mse = np.mean((realized_vol - predicted_vol) ** 2)
    qlike = np.mean(np.log(predicted_var) + realized_var / predicted_var)
    return mse, qlike

def get_ml_predictions_rolling(model, history_returns, test_returns, seq_len):
    preds = []
    used_returns = []
    with torch.no_grad():
        for r in test_returns:
            if len(history_returns) < seq_len:
                history_returns.append(r)
                continue
            seq = torch.tensor(history_returns[-seq_len:], dtype=torch.float32, device=device).unsqueeze(0)
            _, var_seq = model(seq)
            preds.append(var_seq[:, -1].sqrt().cpu().numpy()[0])
            used_returns.append(r)
            history_returns.append(r)
    return np.array(preds), np.array(used_returns)

def train_lstm_garch_for_series(name, series, splits, epochs=100):
    if name not in splits.index:
        return None
    s = series.dropna().values.astype(np.float32)
    train_len = int(splits.loc[name, "Train"])
    val_len = int(splits.loc[name, "Val"])
    test_len = int(splits.loc[name, "Test"])
    if train_len + val_len + test_len > len(s) or train_len <= seq_len or test_len <= seq_len:
        return None
    train = s[:train_len]
    val = s[train_len:train_len + val_len]
    test = s[train_len + val_len:train_len + val_len + test_len]
    train_seq = create_sequences(train, seq_len)
    if len(train_seq) == 0:
        return None
    train_tensor = torch.tensor(train_seq, dtype=torch.float32)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_tensor, batch_size=64, shuffle=True, pin_memory=pin)
    model = LSTMGARCH().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    for _ in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device, non_blocking=True)
            opt.zero_grad()
            loss, _ = model(batch)
            loss.backward()
            opt.step()
    history = list(train)
    ml_vol, returns_eval = get_ml_predictions_rolling(model, history, test.tolist(), seq_len)
    if len(ml_vol) == 0:
        return None
    realized_vol = np.abs(returns_eval)
    realized_var = returns_eval ** 2
    ml_var = ml_vol ** 2
    mse, qlike = compute_metrics(realized_vol, ml_vol, realized_var, ml_var)
    var_calc = var_calculator.RollingVaRCalculator(confidence_level=0.95, min_history=30, refit_frequency=5)
    var_estimates = var_calc.compute_var_rolling_window(returns_eval, ml_vol, confidence_level=0.95)
    valid_mask = (~np.isnan(var_estimates)) & (~np.isnan(returns_eval))
    if not valid_mask.any():
        return None
    returns_valid = returns_eval[valid_mask]
    vol_valid = ml_vol[valid_mask]
    var_valid = var_estimates[valid_mask]
    viol_rate, kupiec_lr, kupiec_p = var_calculator.compute_kupiec_test(returns_valid, var_valid, confidence_level=0.95)
    traffic_light, cum_violations = var_calculator.compute_traffic_light_test(returns_valid, var_valid, confidence_level=0.95)
    lr_ind, p_ind = var_calculator.compute_christoffersen_independence(returns_valid, var_valid, confidence_level=0.95)
    return {
        "Dataset": name,
        "MSE": f"{mse:.6f}",
        "QLIKE": f"{qlike:.6f}",
        "Violation_Rate": f"{viol_rate:.4f}",
        "Kupiec_LR": f"{kupiec_lr:.4f}" if not np.isnan(kupiec_lr) else "N/A",
        "Kupiec_p": f"{kupiec_p:.4f}" if not np.isnan(kupiec_p) else "N/A",
        "LR_Ind": f"{lr_ind:.4f}" if not np.isnan(lr_ind) else "N/A",
        "p_Ind": f"{p_ind:.4f}" if not np.isnan(p_ind) else "N/A",
        "Traffic_Light": traffic_light,
        "Cum_Violations": int(cum_violations) if not np.isnan(cum_violations) else "N/A",
        "Mean_Nu": "N/A",
    }


In [16]:
# Train + test LSTM-GARCH across datasets
results = []
for name, series in datasets.items():
    res = train_lstm_garch_for_series(name, series, split_df, epochs=200)
    if res is not None:
        results.append(res)

if results:
    results_df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("LSTM-GARCH RESULTS")
    print("="*80)
    print(results_df)
    print("="*80)
else:
    print("No results generated (check splits/sequence length).")

NameError: name 'datasets' is not defined

In [35]:
results_df

,Dataset,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p,LR_Ind,p_Ind,Traffic_Light,Cum_Violations,Mean_Nu
0,Stock returns,1.353511,1.353553,0.0089,18.0210,0.0000,0.0539,0.8164,Green,3,3.4665
1,VN30 Index,0.022198,-3.699206,0.0000,N/A,N/A,N/A,N/A,Green,0,6.8072
2,VN Index,1.309424,1.279683,0.0129,12.6836,0.0004,0.1046,0.7464,Green,4,5.8786
3,DAX_40,0.926853,1.171321,0.0000,N/A,N/A,N/A,N/A,Green,0,3.5116
4,EuroNext_100,0.740781,0.783983,0.0103,28.4499,0.0000,11.4792,0.0007,Green,5,20.0642
5,IBEX_35,0.903265,1.063701,0.0102,19.0486,0.0000,4.9790,0.0257,Green,3,14.3045
6,KOSPI_index,1.305005,1.599496,0.0133,11.9209,0.0006,0.1081,0.7423,Green,4,4.3246
7,SMI,0.909533,0.843222,0.0156,8.6806,0.0032,4.1605,0.0414,Green,3,5.0712
8,snp500,1.447073,1.180534,0.0154,6.6711,0.0098,4.9343,0.0263,Green,3,4.0872
9,Nikkei_225,1.763598,1.659140,0.0178,6.4706,0.0110,0.1455,0.7029,Green,4,7.2275


In [8]:
data = {
    "Dataset": [
        "Stock returns", "VN30 Index", "VN Index", "DAX_40",
        "EuroNext_100", "IBEX_35", "KOSPI_index", "SMI",
        "snp500", "Nikkei_225"
    ],
    "MSE": [3.645909, 0.000226, 2.156749, 1.794767, 2.043014, 4.350714, 1.603690, 0.972589, 0.702171, 2.522052],
    "QLIKE": [1.791973, -7.476546, 1.430401, 1.298465, 1.072953, 1.772669, 1.695694, 0.667386, 0.624339, 1.785674],
    "Violation_Rate": [0.0506, 0.0401, 0.0676, 0.0159, 0.0162, 0.0083, 0.0554, 0.0441, 0.0364, 0.0205],
    "Kupiec_LR": [0.0018, 0.5989, 1.6614, 4.1630, 17.8908, 19.9838, 0.1581, 0.1758, 0.7107, 4.5489],
    "Kupiec_p": [0.9658, 0.4390, 0.1974, 0.0413, 0.0000, 0.0000, 0.6909, 0.6750, 0.3992, 0.0329]
}

results_df = pd.DataFrame(data)

In [10]:
print(results_df)

         Dataset       MSE     QLIKE  Violation_Rate  Kupiec_LR  Kupiec_p
0  Stock returns  3.645909  1.791973          0.0506     0.0018    0.9658
1     VN30 Index  0.000226 -7.476546          0.0401     0.5989    0.4390
2       VN Index  2.156749  1.430401          0.0676     1.6614    0.1974
3         DAX_40  1.794767  1.298465          0.0159     4.1630    0.0413
4   EuroNext_100  2.043014  1.072953          0.0162    17.8908    0.0000
5        IBEX_35  4.350714  1.772669          0.0083    19.9838    0.0000
6    KOSPI_index  1.603690  1.695694          0.0554     0.1581    0.6909
7            SMI  0.972589  0.667386          0.0441     0.1758    0.6750
8         snp500  0.702171  0.624339          0.0364     0.7107    0.3992
9     Nikkei_225  2.522052  1.785674          0.0205     4.5489    0.0329


Other models

In [9]:

import sys
import importlib
from pathlib import Path

# ==============================================================================
# 6. Run the pipeline
# ==============================================================================
# Ensure project paths are available for imports
workspace_root = Path.cwd()
if not (workspace_root / "ultility").exists():
    workspace_root = workspace_root.parent
model_dir = workspace_root / "model"

for p in [workspace_root, model_dir]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ultility.data_loader
import ultility.metrics
import ultility.models_garch
import ultility.models_transformer
import train_pipeline

importlib.reload(ultility.data_loader)
importlib.reload(ultility.metrics)
importlib.reload(ultility.models_garch)
importlib.reload(ultility.models_transformer)
importlib.reload(train_pipeline)

print("Starting the benchmarking pipeline...")
#results_df = train_pipeline.run_benchmark(datasets, split_df, seq_len=60)

print("\\n" + "="*80)
print("BASELINE MODELS BENCHMARKING RESULTS")
print("="*80)
if "results_df" in globals():
    display(results_df)
else:
    print("results_df is not available. Run previous cells first or uncomment benchmark line.")
print("="*80)


Starting the benchmarking pipeline...
\n================================================================================
BASELINE MODELS BENCHMARKING RESULTS


,Dataset,MSE,QLIKE,Violation_Rate,Kupiec_LR,Kupiec_p
0,Stock returns,3.645909,1.791973,0.0506,0.0018,0.9658
1,VN30 Index,0.000226,-7.476546,0.0401,0.5989,0.4390
2,VN Index,2.156749,1.430401,0.0676,1.6614,0.1974
3,DAX_40,1.794767,1.298465,0.0159,4.1630,0.0413
4,EuroNext_100,2.043014,1.072953,0.0162,17.8908,0.0000
5,IBEX_35,4.350714,1.772669,0.0083,19.9838,0.0000
6,KOSPI_index,1.603690,1.695694,0.0554,0.1581,0.6909
7,SMI,0.972589,0.667386,0.0441,0.1758,0.6750
8,snp500,0.702171,0.624339,0.0364,0.7107,0.3992
9,Nikkei_225,2.522052,1.785674,0.0205,4.5489,0.0329


In [10]:



# Ensure project paths are available for imports
workspace_root = Path.cwd()
if not (workspace_root / "ultility").exists():
    workspace_root = workspace_root.parent
model_dir = workspace_root / "model"

for p in [workspace_root, model_dir]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))


print("Starting the benchmarking pipeline...")
results_df = train_pipeline.run_benchmark(datasets, split_df, seq_len=60)

print("\\n" + "="*80)
print("BASELINE MODELS BENCHMARKING RESULTS")
print("="*80)
display(results_df)
print("="*80)


Starting the benchmarking pipeline...


Processing Datasets:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch [10/80], Train Loss: 0.910029, Val Loss: 0.442500
Epoch [20/80], Train Loss: 0.896495, Val Loss: 0.444019
Epoch [30/80], Train Loss: 0.873511, Val Loss: 0.450424
Epoch [40/80], Train Loss: 0.833010, Val Loss: 0.464840
Epoch [50/80], Train Loss: 0.791941, Val Loss: 0.465928
Epoch [60/80], Train Loss: 0.753487, Val Loss: 0.475552
Epoch [70/80], Train Loss: 0.731028, Val Loss: 0.472403
Epoch [80/80], Train Loss: 0.664656, Val Loss: 0.466464


Processing Datasets:  10%|█         | 1/10 [2:18:26<20:46:00, 8306.74s/it]


KeyboardInterrupt: 

In [ ]:
results_df.to_csv('results.csv')